In [ ]:
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.listdir("/content/drive/My Drive/LBP/LPB")

['LBP']

In [ ]:
!pytest "/content/drive/MyDrive/LBP/LPB/LBP/tests/test_classify.py"

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/LBP/LPB/LBP
configfile: pytest.ini
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 6 items                                                              

drive/MyDrive/LBP/LPB/LBP/tests/test_classify.py ......                  [100%]

============================== 6 passed in 11.72s ==============================


In [ ]:
!pytest "/content/drive/MyDrive/LBP/LPB/LBP/tests/test_classify.py"

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/LBP/LPB/LBP
configfile: pytest.ini
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 6 items                                                              

drive/MyDrive/LBP/LPB/LBP/tests/test_classify.py ......                  [100%]

============================== 6 passed in 0.17s ===============================


In [ ]:
!pytest "/content/drive/MyDrive/LBP/LPB/LBP/tests/test_distances.py"

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/drive/MyDrive/LBP/LPB/LBP
configfile: pytest.ini
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collected 12 items                                                             

drive/MyDrive/LBP/LPB/LBP/tests/test_distances.py ............           [100%]

============================== 12 passed in 0.70s ==============================


In [ ]:
!PYTHONPATH="/content/drive/MyDrive/LBP/LPB/LBP" python "/content/drive/MyDrive/LBP/LPB/LBP/experiments/orl.py" --data /content/drive/MyDrive/LBP/LPB/LBP/data/ORL --seed 0 --out results/orl_seed0.json

ORL: 390 images, feature length 2187, features+distances 7.5s
mean recognition rate 0.9833  std 0.0106  95% [0.959, 1.000]  (100 permutations)
paper reports         0.9800  std 0.0120
wrote results/orl_seed0.json   digest 74f28a22752e1939...


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/LBP/LPB/LBP")
import numpy as np
from scipy import ndimage
from lbpface.datasets import load_orl
from lbpface.features import LBPDescriptor
from lbpface.distances import pairwise_distances
from lbpface.classify import recognition_rate

fs = load_orl("/content/drive/MyDrive/LBP/LPB/LBP/data/ORL")
desc = LBPDescriptor.orl()                          # LBP(16,2), 30x37 windows
is_gallery = (np.arange(len(fs.images)) % 10) < 5   # images 1-5 of each subject
gal, prb = fs.images[is_gallery], fs.images[~is_gallery]
gl, pl = fs.subjects[is_gallery], fs.subjects[~is_gallery]
G = desc.transform(gal)

def score(probe_images):
    P = desc.transform(probe_images)
    return recognition_rate(pairwise_distances(P, G, "chi2"), gl, pl)

def illum_ramp(img, s):          # darker on the left, brighter on the right
    return np.clip(img * (1 + s * np.linspace(-1, 1, img.shape[1])), 0, 255)

def rotate(img, deg):
    return ndimage.rotate(img.astype(float), deg, reshape=False, order=1, mode="nearest")

print("baseline           ", score(prb))
print("uniform brightness ", score([0.5 * i + 20 for i in prb]))
for s in [0.2, 0.4, 0.6, 0.8]: print("lighting ramp", s, score([illum_ramp(i, s) for i in prb]))
for d in [5, 10, 20]:          print("rotate", d, "deg", score([rotate(i, d) for i in prb]))


baseline            0.9743589743589743
uniform brightness  0.9743589743589743
lighting ramp 0.2 0.958974358974359
lighting ramp 0.4 0.9487179487179487
lighting ramp 0.6 0.8974358974358975
lighting ramp 0.8 0.7025641025641025
rotate 5 deg 0.9692307692307692
rotate 10 deg 0.9282051282051282
rotate 20 deg 0.6307692307692307


In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/LBP/LPB/LBP")

import numpy as np
from scipy import ndimage
from lbpface.datasets import load_orl
from lbpface.features import LBPDescriptor
from lbpface.distances import pairwise_distances
from lbpface.classify import recognition_rate

fs = load_orl("/content/drive/MyDrive/LBP/LPB/LBP/data/ORL")
desc = LBPDescriptor.orl()                          # LBP(16,2), 30x37 windows
is_gallery = (np.arange(len(fs.images)) % 10) < 5   # images 1-5 of each subject
gal, prb = fs.images[is_gallery], fs.images[~is_gallery]
gl, pl = fs.subjects[is_gallery], fs.subjects[~is_gallery]

# Advanced simple normalization: Zero-Mean Unit-Variance (Z-score) standardization
def normalize_image(img):
    img_flat = img.astype(float)
    mean = np.mean(img_flat)
    std = np.std(img_flat)
    if std > 0:
        normalized = (img_flat - mean) / std
    else:
        normalized = img_flat - mean

    # Scale back to 0-255 range for LBP descriptor extraction
    n_min, n_max = normalized.min(), normalized.max()
    if n_max > n_min:
        return (255.0 * (normalized - n_min) / (n_max - n_min)).astype('uint8')
    return np.zeros_like(img, dtype='uint8')

# Normalize all images
gal_norm = np.array([normalize_image(i) for i in gal])

G = desc.transform(gal_norm)

def score(probe_images):
    prb_norm = np.array([normalize_image(i) for i in probe_images])
    P = desc.transform(prb_norm)
    return recognition_rate(pairwise_distances(P, G, "chi2"), gl, pl)

def illum_ramp(img, s):          # darker on the left, brighter on the right
    return np.clip(img * (1 + s * np.linspace(-1, 1, img.shape[1])), 0, 255)

def rotate(img, deg):
    return ndimage.rotate(img.astype(float), deg, reshape=False, order=1, mode="nearest")

print("baseline           ", score(prb))
print("uniform brightness ", score([0.5 * i + 20 for i in prb]))
for s in [0.2, 0.4, 0.6, 0.8]: print("lighting ramp", s, score([illum_ramp(i, s) for i in prb]))
for d in [5, 10, 20]:          print("rotate", d, "deg", score([rotate(i, d) for i in prb]))

baseline            0.9794871794871794
uniform brightness  0.9794871794871794
lighting ramp 0.2 0.958974358974359
lighting ramp 0.4 0.9435897435897436
lighting ramp 0.6 0.8923076923076924
lighting ramp 0.8 0.7025641025641025
rotate 5 deg 0.9743589743589743
rotate 10 deg 0.9230769230769231
rotate 20 deg 0.6410256410256411
